## Imports

In [9]:
import pandas as pd
import numpy as np
import os

## Data Collection

In [ ]:
data_dir = "../Data"

stores_df = pd.read_csv(os.path.join(data_dir, 'stores.csv'))
readings_df = pd.read_csv(os.path.join(data_dir, 'energy_readings.csv'))
bills_df = pd.read_csv(os.path.join(data_dir, 'utility_bills.csv'))

## Data Cleaning and Feature Engineering 
#### Merging all csv files into single table using store id as primary key and inner join

In [10]:
# 1. Merge and Deduplicate (solving the double-billing issue programmatically)
merged_df = pd.merge(readings_df, stores_df, on='store_id', how='inner')
full_df = pd.merge(merged_df, bills_df, on='store_id', how='inner').drop_duplicates()

In [15]:
# 2. Group by store to calculate metrics and Z-scores
store_summary = full_df.groupby(['store_id', 'store_type', 'city']).agg({
    'electricity_kwh': 'sum',
    'total_amount': 'sum'
}).reset_index()

In [18]:
# Calculate Z-Scores for anomaly detection
mean_energy = store_summary['electricity_kwh'].mean()
std_energy = store_summary['electricity_kwh'].std()
store_summary['z_score'] = (store_summary['electricity_kwh'] - mean_energy) / std_energy

In [25]:
# Filter High-Risk Outliers
high_risk = store_summary[store_summary['z_score'] > 1.5]

In [26]:
# 3. Audit Pending Payments
pending_bills = bills_df[bills_df['payment_status'].str.lower() == 'pending']

In [27]:
print("=== AUTOMATED PYTHON AUDIT REPORT ===")
print(f"Total Stores Analyzed: {len(stores_df)}")
print(f"High-Risk Outliers Flagged: {len(high_risk)}")
print(f"Total Pending Bills: {len(pending_bills)}")
print(f"Total Pending Amount: ₹{pending_bills['total_amount'].sum():,.2f}")

=== AUTOMATED PYTHON AUDIT REPORT ===
Total Stores Analyzed: 60
High-Risk Outliers Flagged: 7
Total Pending Bills: 47
Total Pending Amount: ₹30,813,178.33
